In [30]:
import time
import math

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [31]:
MODEL_PATH = "face_landmarker.task"
CAMERA_ID = 0

IRIS_L = 468
IRIS_R = 473


BEAM_OPENING_DEG = 20
BEAM_LENGTH = 4
BEAM_COLOR = (0, 200, 255)  # BGR
BEAM_THICKNESS = 3

In [32]:
def create_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options =  vision.FaceLandmarkerOptions(
        base_options = base_options,
        running_mode = vision.RunningMode.VIDEO,
        num_faces = 1
    )
    return vision.FaceLandmarker.create_from_options(options)

In [33]:
def get_eyes(result, w, h):
    if not result.face_landmarks:
        return None

    points = result.face_landmarks[0]

    left = points[IRIS_L]
    right = points[IRIS_R]

    if right.x < left.x:
        left, right = right, left 

    left_px = (left.x * w, left.y * h)
    right_px = (right.x * w, right.y * h)

    return left_px, right_px


In [34]:
def to_int(p):
    return(int(round(p[0])), int(round(p[1])))

In [35]:
def beam_end(eye, angle, length):
    return (
        eye[0] + length * math.cos(angle),
        eye[1] + length * math.sin(angle)
    )

In [36]:
detector = create_detector()

v = cv2.VideoCapture(CAMERA_ID)

t_0 = time.perf_counter()

while True:
    ret, frame = v.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp = int((time.perf_counter() - t_0) * 1000)
    r = detector.detect_for_video(mp_img, timestamp)

    eyes = get_eyes(r, w, h)
    if eyes is not None:
        eye_l, eye_r = eyes
        
        eye_v = (eye_r[0] - eye_l[0], eye_r[1] - eye_l[1])
        eye_mod = math.hypot(eye_v[0], eye_v[1])
        theta = math.atan2(eye_v[1], eye_v[0])

        phi = math.radians(BEAM_OPENING_DEG)
        down = theta + math.pi / 2
        angle_l = down + phi
        angle_r = down - phi

        length = BEAM_LENGTH * eye_mod

        end_l = beam_end(eye_l, angle_l, length)
        end_r = beam_end(eye_r, angle_r, length)

        cv2.line(
             frame, to_int(eye_l), to_int(end_l),
             BEAM_COLOR, BEAM_THICKNESS, cv2.LINE_AA
        )
        cv2.line(
             frame, to_int(eye_r), to_int(end_r),
             BEAM_COLOR, BEAM_THICKNESS, cv2.LINE_AA
        )

        """
        for center in eyes:
            cv2.circle(frame, center, 6, (0, 255, 255), 2)
            cv2.circle(frame, center, 1, (0, 0, 255), -1)

        cv2.line(frame, eyes[0], eyes[1], (0, 255, 0), 1)
        """
    else:
        cv2.putText(
            frame, "Sin cara", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
        )

    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
            break
    
v.release()
cv2.destroyAllWindows()
detector.close()

W0000 00:00:1795167102.002850   44421 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1795167102.016077   44424 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795167102.042284   44424 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
